# Fine-tune BERTimbau → Hugging Face (Parliamentary Auditor)

Notebook **único** para:
1. carregar e limpar o corpus parlamentar
2. fazer fine-tuning do **BERTimbau** (`neuralmind/bert-base-portuguese-cased`)
3. salvar tokenizer + modelo no formato Hugging Face (`save_pretrained`)
4. (opcional) enviar ao Hub com `push_to_hub`

**Taxonomia (4 classes)** — alinhada ao MCP (`parliamentary_nlp.model.LABELS`):

| id | Label |
|----|--------|
| 0 | `NEUTRAL` |
| 1 | `GENERIC_OFFENSE` |
| 2 | `TARGETED_OFFENSE` |
| 3 | `EXPLICIT_HATE_SPEECH` |

Protocolo alinhado aos experimentos: AdamW `2e-5`, batch 8, max length 128, Focal Loss (γ=2) + WeightedRandomSampler, early stopping por Macro-F1.


## 1) Instalação


In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("IN_COLAB:", IN_COLAB)

if IN_COLAB:
    !nvidia-smi -L || echo "Sem GPU — Runtime > Change runtime type > GPU"
    !pip -q install -U "transformers>=4.38" "accelerate" scikit-learn openpyxl seaborn
    !pip -q install "pandas==2.2.2"
    # torch já vem no Colab
else:
    print("Local: garanta torch + transformers + scikit-learn + pandas + openpyxl instalados.")


## 2) Imports e configuração


In [ ]:
from __future__ import annotations

import copy
import json
import os
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, f1_score, matthews_corrcoef
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

# ---------------------------------------------------------------------------
# Config — ajuste aqui
# ---------------------------------------------------------------------------
SEED = 42
BASE_MODEL = "neuralmind/bert-base-portuguese-cased"
DATA_URL = (
    "https://raw.githubusercontent.com/alissonf216/"
    "deteccao_discurso_odio_deputados/main/dados/"
    "discursos_deputados_classificados.xlsx"
)

BATCH_SIZE = 8
EPOCHS = 12
PATIENCE = 3
LR = 2e-5
MAX_LENGTH = 128
FOCAL_GAMMA = 2.0
USE_SAMPLER = True
VAL_SIZE = 0.15
TEST_SIZE = 0.15

# Pasta local no formato HF (pronta para upload)
OUTPUT_DIR = Path("outputs/parliamentary-bertimbau-auditor")

# Hub (opcional) — ex.: "seu-usuario/parliamentary-bertimbau-auditor"
HF_REPO_ID = ""  # deixe vazio para só salvar local
HF_PRIVATE = False
HF_TOKEN = os.environ.get("HF_TOKEN")  # ou cole o token na célula de push

ID2LABEL = {
    0: "NEUTRAL",
    1: "GENERIC_OFFENSE",
    2: "TARGETED_OFFENSE",
    3: "EXPLICIT_HATE_SPEECH",
}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(ID2LABEL)

DESC_TO_LABEL = {
    "discurso neutro ou nao ofensivo": 0,
    (
        "discurso potencialmente ofensivo, mas nao diretamente "
        "relacionado a grupos protegidos"
    ): 1,
    (
        "discurso ofensivo direcionado a grupos protegidos, "
        "mas sem incitacao a violencia"
    ): 2,
    (
        "discurso de odio que incita violencia, odio ou "
        "discriminacao contra grupos protegidos"
    ): 3,
}

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print("device:", device)
print("output:", OUTPUT_DIR.resolve())


## 3) Utilitários (dataset, Focal Loss, seed)


In [ ]:
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_desc(text) -> str:
    if pd.isna(text):
        return ""
    t = str(text).lower()
    for a, b in {
        "á": "a", "à": "a", "ã": "a", "â": "a",
        "é": "e", "ê": "e", "í": "i",
        "ó": "o", "ô": "o", "õ": "o",
        "ú": "u", "ü": "u", "ç": "c",
    }.items():
        t = t.replace(a, b)
    return " ".join(t.split())


class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length: int = MAX_LENGTH):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item


class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, weight=None, reduction: str = "mean"):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=-1)
        probs = log_probs.exp()
        targets = targets.long()
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        loss = -((1.0 - pt) ** self.gamma) * log_pt
        if self.weight is not None:
            loss = loss * self.weight.gather(0, targets)
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


def make_weighted_sampler(labels):
    counts = Counter(int(y) for y in labels)
    class_w = {c: 1.0 / n for c, n in counts.items()}
    sample_w = [class_w[int(y)] for y in labels]
    return WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)


set_seed(SEED)


## 4) Carregar e limpar o corpus


In [ ]:
raw = pd.read_excel(DATA_URL, sheet_name="Dados_completos")
df = raw[["id_deputado", "frase", "label", "label_descricao"]].copy()
df["frase"] = df["frase"].astype(str).str.strip()
df["label"] = df["label"].astype(int)

print("Antes:", len(df))
print(df["label"].value_counts().sort_index())

df["desc_norm"] = df["label_descricao"].map(normalize_desc)
df["label_from_desc"] = df["desc_norm"].map(DESC_TO_LABEL)
mask_fix = df["label_from_desc"].notna() & (df["label"] != df["label_from_desc"])
df.loc[mask_fix, "label"] = df.loc[mask_fix, "label_from_desc"].astype(int)
print(f"Labels corrigidos: {int(mask_fix.sum())}")

n_before = len(df)
df = df.drop_duplicates(subset=["frase"], keep="first").reset_index(drop=True)
print(f"Duplicatas removidas: {n_before - len(df)} | Após limpeza: {len(df)}")
print(df["label"].value_counts().sort_index())
df["label_name"] = df["label"].map(ID2LABEL)
df.head(3)


## 5) Split estratificado (train / val / test)


In [ ]:
texts = df["frase"].tolist()
labels = df["label"].astype(int).tolist()

X_temp, X_test, y_temp, y_test = train_test_split(
    texts,
    labels,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=labels,
)
rel_val = VAL_SIZE / (1.0 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=rel_val,
    random_state=SEED,
    stratify=y_temp,
)

print(f"train={len(X_train)} | val={len(X_val)} | test={len(X_test)}")
print("train dist:", Counter(y_train))
print("val dist:  ", Counter(y_val))
print("test dist: ", Counter(y_test))


## 6) Fine-tuning BERTimbau


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

train_ds = TextDataset(X_train, y_train, tokenizer)
val_ds = TextDataset(X_val, y_val, tokenizer)
test_ds = TextDataset(X_test, y_test, tokenizer)

if USE_SAMPLER:
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, sampler=make_weighted_sampler(y_train)
    )
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(device)

classes = np.unique(y_train)
cw = compute_class_weight("balanced", classes=classes, y=np.asarray(y_train))
weight_vec = np.ones(NUM_LABELS, dtype=np.float32)
for c, w in zip(classes, cw):
    weight_vec[int(c)] = w
class_weights = torch.tensor(weight_vec, dtype=torch.float, device=device)
print("class_weights:", weight_vec)

criterion = FocalLoss(gamma=FOCAL_GAMMA, weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
num_training_steps = max(1, len(train_loader) * EPOCHS)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)

best_f1 = -1.0
best_state = None
patience = 0
history = []

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**{k: v for k, v in batch.items() if k != "labels"})
        loss = criterion(outputs.logits, batch["labels"])
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running += float(loss.item())

    model.eval()
    preds, true = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**{k: v for k, v in batch.items() if k != "labels"})
            pred = torch.argmax(outputs.logits, dim=1)
            preds.extend(pred.cpu().numpy())
            true.extend(batch["labels"].cpu().numpy())
    val_f1 = f1_score(true, preds, average="macro", zero_division=0)
    avg_loss = running / max(1, len(train_loader))
    history.append({"epoch": epoch + 1, "train_loss": avg_loss, "val_macro_f1": val_f1})
    print(f"Epoch {epoch + 1:02d} | loss={avg_loss:.4f} | Val Macro-F1={val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = copy.deepcopy(model.state_dict())
        patience = 0
    else:
        patience += 1
        if patience >= PATIENCE:
            print("Early stopping")
            break

if best_state is not None:
    model.load_state_dict(best_state)
print(f"Melhor Val Macro-F1: {best_f1:.4f}")


## 7) Avaliação no conjunto de teste


In [ ]:
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(**{k: v for k, v in batch.items() if k != "labels"}).logits
        y_pred.extend(torch.argmax(logits, dim=1).cpu().numpy())
        y_true.extend(batch["labels"].cpu().numpy())

target_names = [ID2LABEL[i] for i in range(NUM_LABELS)]
print(classification_report(y_true, y_pred, target_names=target_names, digits=4, zero_division=0))
print("Macro-F1:", f1_score(y_true, y_pred, average="macro", zero_division=0))
print("MCC:     ", matthews_corrcoef(y_true, y_pred))


## 8) Salvar no formato Hugging Face (local)

Gera a pasta com `config.json`, pesos, tokenizer e `id2label` — pronta para o Hub e para o MCP via `PARLIAMENTARY_NLP_MODEL_ID`.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Garante mapeamento de labels no config antes de salvar
model.config.id2label = {int(k): v for k, v in ID2LABEL.items()}
model.config.label2id = LABEL2ID
model.config.problem_type = "single_label_classification"

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

meta = {
    "base_model": BASE_MODEL,
    "task": "parliamentary_offensive_hate_speech",
    "num_labels": NUM_LABELS,
    "id2label": ID2LABEL,
    "label2id": LABEL2ID,
    "max_length": MAX_LENGTH,
    "train_hyperparams": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "focal_gamma": FOCAL_GAMMA,
        "use_sampler": USE_SAMPLER,
        "patience": PATIENCE,
        "seed": SEED,
    },
    "best_val_macro_f1": float(best_f1),
    "history": history,
}
(OUTPUT_DIR / "training_meta.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8"
)

# README curto no card do modelo (opcional no Hub)
card = f"""---
language: pt
tags:
  - text-classification
  - bertimbau
  - hate-speech
  - parliamentary
  - portuguese
license: mit
---

# Parliamentary BERTimbau Auditor

Fine-tuned [`{BASE_MODEL}`](https://huggingface.co/{BASE_MODEL}) for 4-class offensive / hate-speech detection in Brazilian parliamentary discourse.

## Labels

| id | label |
|----|--------|
| 0 | NEUTRAL |
| 1 | GENERIC_OFFENSE |
| 2 | TARGETED_OFFENSE |
| 3 | EXPLICIT_HATE_SPEECH |

## Usage

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

repo = "{OUTPUT_DIR.name}"  # or your HF repo id
tok = AutoTokenizer.from_pretrained(repo)
model = AutoModelForSequenceClassification.from_pretrained(repo)
text = "Senhor presidente, peço a palavra."
inputs = tok(text, return_tensors="pt", truncation=True, max_length=128)
with torch.no_grad():
    probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
pred = int(probs.argmax())
print(model.config.id2label[pred], float(probs[pred]))
```

Best validation Macro-F1 (this run): **{best_f1:.4f}**
"""
(OUTPUT_DIR / "README.md").write_text(card, encoding="utf-8")

print("Salvo em:", OUTPUT_DIR.resolve())
print("Arquivos:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUTPUT_DIR)}  ({p.stat().st_size/1e6:.2f} MB)")


## 9) Smoke test do checkpoint salvo

Confirma que o diretório carrega como o MCP fará (`Auto*` + `id2label`).


In [ ]:
tok2 = AutoTokenizer.from_pretrained(OUTPUT_DIR)
mdl2 = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
mdl2.eval()

samples = [
    "O debate deve ser respeitoso e baseado em evidências.",
    "Esse parlamentar é um corrupto incompetente e não merece ocupar a cadeira.",
]
for text in samples:
    inputs = tok2(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    with torch.no_grad():
        probs = torch.softmax(mdl2(**inputs).logits, dim=-1).squeeze(0)
    pred = int(probs.argmax())
    label = mdl2.config.id2label[pred]
    print(f"[{label} | {float(probs[pred]):.4f}] {text}")


## 10) (Opcional) Push para o Hugging Face Hub

1. Crie um token em https://huggingface.co/settings/tokens  
2. Defina `HF_REPO_ID` (ex.: `alissonf216/parliamentary-bertimbau-auditor`)  
3. Rode a célula abaixo


In [ ]:
# Preencha antes de rodar:
# HF_REPO_ID = "seu-usuario/parliamentary-bertimbau-auditor"
# HF_TOKEN = "hf_..."   # ou exporte HF_TOKEN no ambiente

if not HF_REPO_ID:
    print("Defina HF_REPO_ID para enviar ao Hub. Checkpoint local já está pronto.")
else:
    from huggingface_hub import login

    token = HF_TOKEN or os.environ.get("HF_TOKEN")
    if not token:
        raise ValueError("Defina HF_TOKEN (variável de ambiente ou na célula).")

    login(token=token)

    model.push_to_hub(HF_REPO_ID, private=HF_PRIVATE, token=token)
    tokenizer.push_to_hub(HF_REPO_ID, private=HF_PRIVATE, token=token)

    # sobe o model card
    from huggingface_hub import HfApi

    api = HfApi(token=token)
    readme = OUTPUT_DIR / "README.md"
    if readme.exists():
        api.upload_file(
            path_or_fileobj=str(readme),
            path_in_repo="README.md",
            repo_id=HF_REPO_ID,
            repo_type="model",
        )
    print(f"Publicado: https://huggingface.co/{HF_REPO_ID}")
    print("No MCP:")
    print(f'  export PARLIAMENTARY_NLP_MODEL_ID="{HF_REPO_ID}"')


## 11) Usar no MCP

Depois do upload (ou com a pasta local):

```bash
# Hub
export PARLIAMENTARY_NLP_MODEL_ID="seu-usuario/parliamentary-bertimbau-auditor"

# ou pasta local
export PARLIAMENTARY_NLP_MODEL_ID="/caminho/absoluto/outputs/parliamentary-bertimbau-auditor"

parliamentary-nlp-mcp
```

Reinicie o Inspector / Cursor após trocar o id.
